<a href="https://colab.research.google.com/github/knayoung132-collab/voice-phishing-detection/blob/main/%EB%8B%A8%EC%96%B4%EC%82%AC%EC%A0%84%EC%BD%94%EB%93%9C.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os, re, json, unicodedata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter

In [ ]:
# Replace '/path/to/your/file.txt' with the actual path to your file in Google Drive
file_path = '/content/drive/MyDrive/보이스피싱_텍스트데이터/all_preprocessed.csv'

# Example: Read a text file
# try:
#     with open(file_path, 'r') as f:
#         content = f.read()
#         print(content)
# except FileNotFoundError:
#     print(f"Error: File not found at {file_path}")
# except Exception as e:
#     print(f"An error occurred: {e}")

# If you are loading a different type of file (e.g., CSV, Excel),
# you might need to use a library like pandas.
import pandas as pd
try:
    df = pd.read_csv(file_path)
    display(df.head())
except FileNotFoundError:
    print(f"Error: File not found at {file_path}")
except Exception as e:
    print(f"An error occurred: {e}")

,text,clean_text,source_file
0,text,NaN,사이버경찰청_수사관_사칭_감기오셨어요_ᄇ...
1,서울 지방경찰청에 있는 사이버 수사대입니다.,서울 지방 경찰청 있는 사이버 수사 대 입니다,사이버경찰청_수사관_사칭_감기오셨어요_ᄇ...
2,"예, 그래서요?",예 그래서요,사이버경찰청_수사관_사칭_감기오셨어요_ᄇ...
3,다름이 아니고요. 지금 저희가 수사 중에 본인 앞으로 연루된 사건이 있어서 전화 드...,다름이 아니고요 지금 저희 수사 중 본인 앞 으로 연루 된 사건 있어서 전화 드린 겁니다,사이버경찰청_수사관_사칭_감기오셨어요_ᄇ...
4,"예, 왜요?",예 왜 요,사이버경찰청_수사관_사칭_감기오셨어요_ᄇ...


In [ ]:
# =========================
# 0) 경로 & 설정
# =========================
file1 = "/content/drive/MyDrive/중간평가_이후_모델/all_preprocessed.csv"   # text,label
file2 = "/content/drive/MyDrive/중간평가_이후_모델/merged_dialogues.csv"  # text만, 모두 0(일반)

text_col  = "text"
label_col = "label"
pos_label = 1

out_dir = "/content/drive/MyDrive/중간평가_이후_모델/lexicon_artifacts"
os.makedirs(out_dir, exist_ok=True)

# 빠른 튜닝 포인트
TOP_K = 30            # LEXICON 넣을 상위 개수(20~30 추천)
CHUNK  = 100_000      # file2 청크 크기
W_NEG  = 0.3          # 불균형 보정(음성 토큰 가중, 0.2~0.5 탐색)
ALPHA  = 0.05         # 스무딩
MIN_LEN = 2           # 토큰 최소 길이
MIN_POS_DOC = 5       # 양성 문서 최소 등장(희귀어 제거)
LOW_DISCRIM_Z = 0.2   # |log-odds|가 작으면 구분력 낮다고 간주(Stop 후보)

# 기본 Stopwords (너무 일반적인 단어들)
BASE_STOP = {
    "안내","고객님","확인","연락","상담","문의","서비스","이용","감사","공지",
    "처리","발송","등록","접수","정보","알림","요청","사용","가능","제공"
}

In [ ]:
# =========================
# 1) 정규화 & 토큰화
# =========================
def normalize_ko(s: str) -> str:
    s = unicodedata.normalize("NFKC", str(s))
    s = re.sub(r"[^\w\s가-힣.,:;!?@#\-/]", " ", s)  # 이모지/특수문자 제거
    s = re.sub(r"\s+", " ", s).strip()
    # 흔한 우회 교정(필요시 확장)
    s = s.replace("계 좌", "계좌").replace("계.좌", "계좌").replace("앱  설치", "앱 설치")
    return s

def word_tokens(text: str):
    return [t for t in re.split(r"\s+", text) if len(t) >= MIN_LEN]


In [ ]:
# 2) 카운트 유틸(TF/DF)
# =========================
def update_tf_df_from_series(series, tf: Counter, df: Counter):
    for s in series.dropna().astype(str):
        toks = word_tokens(normalize_ko(s))
        if not toks:
            continue
        tf.update(toks)
        df.update(set(toks))

def count_file1_pos_neg(file1, text_col, label_col, pos_label):
    df = pd.read_csv(file1, usecols=[text_col, label_col])
    pos_tf, pos_df = Counter(), Counter()
    neg_tf, neg_df = Counter(), Counter()

    pos_mask = (df[label_col] == pos_label)
    neg_mask = ~pos_mask

    update_tf_df_from_series(df.loc[pos_mask, text_col], pos_tf, pos_df)
    update_tf_df_from_series(df.loc[neg_mask, text_col], neg_tf, neg_df)
    return pos_tf, pos_df, neg_tf, neg_df, int(pos_mask.sum()), int(neg_mask.sum())

def count_file2_neg_streaming(file2, text_col, chunksize=CHUNK):
    tf, df = Counter(), Counter()
    total = 0
    for chunk in pd.read_csv(file2, usecols=[text_col], chunksize=chunksize):
        update_tf_df_from_series(chunk[text_col], tf, df)
        total += len(chunk)
    return tf, df, total


In [ ]:
# =========================
# 3) 카운트 수행
# =========================
print("📥 file1 읽는 중...")
pos_tf_1, pos_df_1, neg_tf_1, neg_df_1, n_pos_1, n_neg_1 = count_file1_pos_neg(file1, text_col, label_col, pos_label)
print(f" - file1: pos={n_pos_1}, neg={n_neg_1}")

print("📥 file2(음성만, 대용량) 스트리밍 카운트 중...")
neg_tf_2, neg_df_2, n_neg_2 = count_file2_neg_streaming(file2, text_col, CHUNK)
print(f" - file2: neg={n_neg_2}")

# 음성 통합
neg_tf = neg_tf_1 + neg_tf_2
neg_df = neg_df_1 + neg_df_2

# 전역(보고용)
all_tf = pos_tf_1 + neg_tf
all_df = pos_df_1 + neg_df


📥 file1 읽는 중...
 - file1: pos=1515, neg=325
📥 file2(음성만, 대용량) 스트리밍 카운트 중...
 - file2: neg=728281


In [ ]:

# =========================
# 4) 보고용 Top-N 데이터프레임
# =========================
def top_table(tf: Counter, df: Counter, topn=100):
    rows = [(t, tf[t], df[t]) for t,_ in tf.most_common(topn)]
    out = pd.DataFrame(rows, columns=["term","tf","df"])
    out.insert(0, "rank", range(1, len(out)+1))
    return out

top_pos = top_table(pos_tf_1, pos_df_1, 100)
top_neg = top_table(neg_tf,    neg_df,    100)
top_all = top_table(all_tf,    all_df,    100)

top_pos_path = os.path.join(out_dir, "top_terms_pos.csv")
top_neg_path = os.path.join(out_dir, "top_terms_neg.csv")
top_all_path = os.path.join(out_dir, "top_terms_global.csv")
top_pos.to_csv(top_pos_path, index=False, encoding="utf-8-sig")
top_neg.to_csv(top_neg_path, index=False, encoding="utf-8-sig")
top_all.to_csv(top_all_path, index=False, encoding="utf-8-sig")
print("🗂️ top_terms_* 저장 완료")

🗂️ top_terms_* 저장 완료


In [ ]:
# =========================
# 5) log-odds 점수 (불균형 보정)
# =========================
def log_odds_scores(pos_tf: Counter, neg_tf: Counter, w_neg=W_NEG, alpha=ALPHA):
    vocab = set(pos_tf) | set(neg_tf)
    pos_total = sum(pos_tf.values()) + alpha*len(vocab)
    neg_total = w_neg*sum(neg_tf.values()) + alpha*len(vocab)
    scores = {}
    for t in vocab:
        a = pos_tf[t] + alpha
        b = w_neg*neg_tf[t] + alpha
        scores[t] = float(np.log(a/(pos_total - a)) - np.log(b/(neg_total - b)))
    return scores

scores = log_odds_scores(pos_tf_1, neg_tf, W_NEG, ALPHA)

In [ ]:
# =========================
# 6) stopword 후보 자동 제안
# =========================
def rank_map(counter: Counter):
    items = counter.most_common()
    return {t: i+1 for i,(t,_) in enumerate(items)}

pos_rank = rank_map(pos_df_1)
neg_rank = rank_map(neg_df)

candidates = []
for t in set(all_df):
    if len(t) < MIN_LEN or re.fullmatch(r"[\d\W_]+", t):
        continue
    s = scores.get(t, 0.0)
    if pos_rank.get(t, 10**9) <= 5000 and neg_rank.get(t, 10**9) <= 5000 and abs(s) < LOW_DISCRIM_Z:
        candidates.append({
            "term": t,
            "pos_df": pos_df_1[t],
            "neg_df": neg_df[t],
            "pos_tf": pos_tf_1[t],
            "neg_tf": neg_tf[t],
            "logodds": s
        })

sw_df = pd.DataFrame(candidates).sort_values(["pos_df","neg_df"], ascending=False)
sw_df = sw_df[sw_df["pos_df"] >= MIN_POS_DOC].reset_index(drop=True)

suggested = BASE_STOP.union(set(sw_df["term"].head(200).tolist()))
stop_json = os.path.join(out_dir, "STOPLIST_suggested.json")
with open(stop_json, "w", encoding="utf-8") as f:
    json.dump(sorted(list(suggested)), f, ensure_ascii=False, indent=2)
sw_csv = os.path.join(out_dir, "stopword_candidates.csv")
sw_df.to_csv(sw_csv, index=False, encoding="utf-8-sig")
print("🛠️ stopword 후보 저장 완료")


🛠️ stopword 후보 저장 완료


In [ ]:
# =========================
# 7) LEXICON: 상위 Top-K 선별 + 가중치 매핑
# =========================
STOP = suggested  # 제안 stoplist 우선 적용(검수 후 업데이트 권장)

cands = []
for t, z in scores.items():
    if len(t) < MIN_LEN or t in STOP or re.fullmatch(r"[\d\W_]+", t):
        continue
    if pos_df_1[t] < MIN_POS_DOC:
        continue
    cands.append((t, z, pos_tf_1[t], neg_tf[t]))

cands.sort(key=lambda x: x[1], reverse=True)
selected = cands[:TOP_K]

LEXICON = {}
for i,(t,z,pt,nt) in enumerate(selected):
    if i < max(1, int(TOP_K*0.2)):   w = 2.5  # 상위 20%
    elif i < max(1, int(TOP_K*0.5)): w = 2.0  # 상위 50%
    else:                            w = 1.5
    LEXICON[t] = float(w)

lex_json = os.path.join(out_dir, "LEXICON_topK_logodds.json")
with open(lex_json, "w", encoding="utf-8") as f:
    json.dump(LEXICON, f, ensure_ascii=False, indent=2)

print(f"✅ LEXICON 저장 완료: {lex_json}")

✅ LEXICON 저장 완료: /content/drive/MyDrive/중간평가_이후_모델/lexicon_artifacts/LEXICON_topK_logodds.json


In [ ]:
# (1) log-odds 상위 20
top_logodds = sorted([(t, z) for t, z in scores.items() if t in dict(selected)], key=lambda x: x[1], reverse=True)[:20]
terms1 = [t for t,_ in top_logodds][::-1]
vals1  = [z for _,z in top_logodds][::-1]

plt.figure(figsize=(8, 7))
plt.barh(terms1, vals1)
plt.title("Top Terms by Log-Odds (Positive-skewed) - Top 20")
plt.xlabel("log-odds score")
plt.ylabel("term")
plt.tight_layout()
plt.show()

ValueError: dictionary update sequence element #0 has length 4; 2 is required